In [6]:
import time
import joblib
import psutil
import pandas as pd
import numpy as np

# --- TEAMMATE'S M3 MODULE ---
class DynamicWorkingSetEstimator:
    def __init__(self, delta=10, fault_threshold=0.30, persistence=2,
                 min_frames=1, max_frames=None):
        self.delta = delta              # observation window (references)
        self.fault_threshold = fault_threshold  # recent fault rate that supports growth
        self.persistence = persistence          # windows a change must persist
        self.min_frames = min_frames
        self.max_frames = max_frames
        self.prev_ws = {}      # pid -> WS(t-1)
        self.need = {}         # pid -> last reported frame need
        self.up_count = {}     # pid -> consecutive growth observations
        self.down_count = {}   # pid -> consecutive shrink observations
        self.info = {}         # pid -> latest diagnostics

    def estimate(self, process_id, references=None, recent_faults=None):
        pid = process_id
        refs = list(references or [])
        window = refs[-self.delta:]                    # steps 1-2
        ws = len(set(window))                          # steps 3-4
        prev = self.prev_ws.get(pid, ws)
        d_ws = ws - prev                               # step 5

        faults = list(recent_faults or [])[-self.delta:]
        fault_rate = sum(faults) / len(faults) if faults else 0.0   # step 6

        need = self.need.get(pid, max(self.min_frames, ws))
        up = self.up_count.get(pid, 0)
        down = self.down_count.get(pid, 0)

        if d_ws > 0:
            up, down = up + 1, 0
        elif d_ws < 0:
            up, down = 0, down + 1

        if ws > need and (fault_rate >= self.fault_threshold or up >= self.persistence):
            need = ws                                  # step 7: grow
        elif ws < need and down >= self.persistence:
            need = max(ws, need - 1)                   # step 8: release gradually
            
        need = max(self.min_frames, need)
        if self.max_frames is not None:
            need = min(self.max_frames, need)

        self.prev_ws[pid], self.need[pid] = ws, need
        self.up_count[pid], self.down_count[pid] = up, down
        self.info[pid] = {"ws": ws, "delta_ws": d_ws,
                         "fault_rate": round(fault_rate, 3), "need": need}
        return need                                    # step 9

# --- INITIALIZE SYSTEM & M3 ---
model = joblib.load("thrashing_detection_model.pkl")
scaler = joblib.load("thrashing_detection_scaler.pkl")
ws_estimator = DynamicWorkingSetEstimator()

old_disk = psutil.disk_io_counters()
print("Starting Integrated System (Detector + M3 Working-Set Manager)...")

try:
    while True:
        # 1. Capture system telemetry (Your Part)
        cpu = psutil.cpu_percent(interval=1)
        mem = psutil.virtual_memory()
        swap = psutil.swap_memory()
        new_disk = psutil.disk_io_counters()
        
        if old_disk and new_disk:
            disk_reads_per_sec = new_disk.read_count - old_disk.read_count
            disk_writes_per_sec = new_disk.write_count - old_disk.write_count
        else:
            disk_reads_per_sec = 0
            disk_writes_per_sec = 0
            
        old_disk = new_disk

        # 2. ML Inference (Your Part)
        features = pd.DataFrame([[
            cpu, mem.percent, swap.percent, disk_reads_per_sec, disk_writes_per_sec
        ]], columns=['cpu_percent', 'ram_percent', 'swap_percent', 'disk_read_count', 'disk_write_count'])
        
        scaled_features = scaler.transform(features)
        print(f"SCALED FEATURES: {scaled_features[0]}")
        probabilities = model.predict_proba(scaled_features)
        print(f"ALL PROBABILITIES: {probabilities[0]}")
        # Adjust order based on your model classes (e.g., Normal, Thrashing, Warning)
        p_normal, p_thrashing, p_warning = probabilities[0]
        
        # 3. State Determination & M3 Handover
        if p_thrashing >= 0.35:
            state = "THRASHING"
            print(f"\n[ALERT] State: THRASHING (Severity: {p_thrashing:.3f})")
            
            # Find heavy memory-consuming processes to pass to M3
            for proc in psutil.process_iter(['pid', 'name', 'memory_percent']):
                try:
                    if proc.info['memory_percent'] and proc.info['memory_percent'] > 5.0: # target heavy processes
                        pid = proc.info['pid']
                        
                        # Mocking or gathering live reference streams / page faults for M3
                        # In a full OS simulator, references/faults come from page table logs.
                        mock_references = [np.random.randint(1, 50) for _ in range(15)]
                        mock_faults = [1 if np.random.rand() > 0.6 else 0 for _ in range(15)]
                        
                        # HANDOVER TO M3: Calculate required working set frames
                        allocated_frames = ws_estimator.estimate(process_id=pid, references=mock_references, recent_faults=mock_faults)
                        print(f" -> M3 Mitigating PID {pid} ({proc.info['name']}): Constraining frames to {allocated_frames}")
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    continue

        elif p_warning >= p_normal and p_warning > p_thrashing:
            state = "WARNING"
            print(f"-> State: WARNING (Monitoring closely...)")
        else:
            state = "NORMAL"
            print(f"-> State: NORMAL")

except KeyboardInterrupt:
    print("\nIntegrated monitoring stopped by user.")

Starting Integrated System (Detector + M3 Working-Set Manager)...
SCALED FEATURES: [-1.3397131  -0.86019429 -0.71523528 -0.81177925 -0.81975885]
ALL PROBABILITIES: [9.58941998e-01 1.54447868e-05 4.10425571e-02]
-> State: NORMAL
SCALED FEATURES: [-1.46829094 -0.86019429 -0.71523528 -0.81238717 -0.83069342]
ALL PROBABILITIES: [9.70586971e-01 9.97959552e-06 2.94030494e-02]
-> State: NORMAL
SCALED FEATURES: [-1.43107104 -0.85543099 -0.71523528 -0.81177925 -0.82522613]
ALL PROBABILITIES: [9.67097078e-01 1.15679722e-05 3.28913536e-02]
-> State: NORMAL
SCALED FEATURES: [-1.42430378 -0.86019429 -0.71523528 -0.71086395 -0.81611399]
ALL PROBABILITIES: [9.68242192e-01 1.48222332e-05 3.17429859e-02]
-> State: NORMAL
SCALED FEATURES: [-1.34309672 -0.86019429 -0.71523528 -0.74004427 -0.84345042]
ALL PROBABILITIES: [9.60357416e-01 1.68574016e-05 3.96257267e-02]
-> State: NORMAL
SCALED FEATURES: [-1.39046751 -0.86019429 -0.71523528 -0.79840493 -0.83069342]
ALL PROBABILITIES: [9.64169510e-01 1.31818823